# Pixels-to-predictions — SmolVLM skeleton **(TA / starter-aligned)**

If the first code cell still shows `os.environ.get("CSV_DIR", ...)` after an update, the editor buffer is stale—**reopen this notebook** or choose **Revert File** so it matches the saved file on disk.

### Changelog (setup / TA pipeline)

- **2026-04:** **Setup cell** — `CSV_DIR`, `DATA_ROOT`, `IMAGE_ROOT`, batching, LR, epochs, etc. are **plain variables** (no env vars). **`os.environ`** is **only** for **`HF_TOKEN`** / **`HUGGING_FACE_HUB_TOKEN`** inside `get_hf_token()`. **`LORA_ADAPTER_FOLDER`** (or `None` → `LORA_SAVE_DIR`) loads a saved adapter; **`HF_FORCE_REDOWNLOAD`** toggles Hub `force_download` in the LoRA cell.
- **2026-04:** **`vqa_dataset` TA prompts** — optional **`Metadata:`** line and alternate answer line via **`TA_INCLUDE_META`** / **`TA_ANSWER_SUFFIX`** in the setup cell; letter parsing accepts **`Answer:`** and **`The correct answer is:`**. Val, smoke, LoRA train, submission, and “Load saved LoRA” all follow those flags.
- **2026-04:** **LoRA (TA PDF tip #4)** — **`LORA_TARGET_MODULES`**: **`q_proj`**, **`v_proj`**, **`gate_proj`**, **`up_proj`**, **`down_proj`**; **`LORA_R=8`**, **`LORA_ALPHA=16`** so trainable params stay **under ~5M** with **DoRA** (~**4.07M** on SmolVLM-500M-Instruct).
- **2026-04:** **Images** — default **`IMG_SIZE=512`** and optional train-only **augmentation** (random resized crop, small rotation, color jitter) via **`IMAGE_AUGMENT`** + **`TaScienceVQADataset(..., image_augment=...)`**; val/test use **`is_train=False`** so eval stays deterministic.
- **2026-05:** **Punnett hybrid** — optional **`data/punnett_allele_overrides.csv`** (`id`, `top1`, `top2`, `left1`, `left2`). When that file has rows, **`run_validation_ta`** / **`predict_test_ta_batched`** call **`hybrid_punnett_predict`** in **`vqa_dataset`** (symbolic 2×2 + question parsing). **`val_preds_ta.csv`** adds **`pred_ta_model`** (LoRA TA index before override) whenever overrides are loaded (see setup print). Optional train flag **`INCLUDE_PUNNETT_ALLELE_HINT`** (setup) appends the parsed allele line from **Hint** on Punnett rows.
- **2026-05:** **Food web (neuro-symbolic)** — **`solve_food_web_symbolic`** in **`vqa_dataset`** matches food-web skill rows to PNG template ids / graph structure and consults **`data/food_web_mc_lookup.json`** (keys: question + sorted choice texts). **`run_validation_ta_batched`** / **`predict_test_ta_batched`** accept **`food_web_symbolic`**; the setup cell defines **`FOOD_WEB_SYMBOLIC`** and the notebook passes it into those calls for easy ablation. When Punnett overrides are loaded *or* food-web symbolic is on, **`val_preds_ta.csv`** also records **`pred_ta_model`** (TA decode before overrides).
Same competition layout as `pixels_vqa_skel.ipynb` (`PROJECT_ROOT`, `CSV_DIR`, `IMAGE_ROOT`, `vqa_dataset.join_p2p_image_path`), but:

- **Prompt** = course starter: `Context:` (lecture then hint), optional metadata, lettered choices, trailing answer line — see `vqa_dataset.build_prompt_ta`. Toggle **`TA_INCLUDE_META`** / **`TA_ANSWER_SUFFIX`** in the **setup cell**.
- **Val** = starter recipe: `max_new_tokens=1`, **full** `batch_decode` of `generated_ids`, letter parse via `extract_ta_choice_letter` (matches ~**0.5429** zero-shot on 500M).

**LoRA fine-tuning:** see the last section — `TaScienceVQADataset` + `collate_vqa_train`; after each epoch, **TA val** (`run_validation_ta`) so the metric matches the starter. Adapter saves to `data/outputs/lora_adapter_ta/`.

In [1]:
import contextlib
import os
import subprocess
import sys
from pathlib import Path

try:
    import httpx
    from huggingface_hub.utils._http import (
        async_hf_request_event_hook,
        async_hf_response_event_hook,
        hf_request_event_hook,
        set_async_client_factory,
        set_client_factory,
    )

    def _hf_httpx_no_br() -> httpx.Client:
        return httpx.Client(
            event_hooks={"request": [hf_request_event_hook]},
            follow_redirects=True,
            timeout=None,
            headers={"Accept-Encoding": "gzip, deflate"},
        )

    def _hf_httpx_async_no_br() -> httpx.AsyncClient:
        return httpx.AsyncClient(
            event_hooks={
                "request": [async_hf_request_event_hook],
                "response": [async_hf_response_event_hook],
            },
            follow_redirects=True,
            timeout=None,
            headers={"Accept-Encoding": "gzip, deflate"},
        )

    set_client_factory(_hf_httpx_no_br)
    set_async_client_factory(_hf_httpx_async_no_br)
except Exception:
    pass


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


IN_COLAB = _in_colab()

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.45.0",
            "peft>=0.11.0",
            "accelerate",
            "safetensors",
            "pandas",
            "Pillow",
            "tqdm",
            "torchao>=0.16.0",  # Changed torchao installation to require version >= 0.16.0
        ],
        stdout=subprocess.DEVNULL,
    )

PROJECT_ROOT = (
    Path("/content/drive/My Drive/DL/final").expanduser().resolve()
    if IN_COLAB
    else Path.cwd().resolve()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Paths (edit if your tree differs; do not use os.environ here) ---
CSV_DIR    = (PROJECT_ROOT / "pixels-to-predictions").resolve()
DATA_ROOT  = (PROJECT_ROOT / "data").resolve()
IMAGE_ROOT = (PROJECT_ROOT / "pixels-to-predictions" / "images" / "images").resolve()

if IN_COLAB:
    from google.colab import userdata
    userdata.get('HF_TOKEN')

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor

try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoModelForImageTextToText


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda", index=0)
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


SEED = 42
torch.manual_seed(SEED)
device = pick_device()
if device.type == "mps":
    torch.set_float32_matmul_precision("medium")
if device.type == "cuda":
    _gp = torch.cuda.get_device_properties(0)
    print(
        f"CUDA device: {torch.cuda.get_device_name(0)}  "
        f"VRAM ~{_gp.total_memory / (1024**3):.1f} GiB"
    )

# --- Model (edit here) ---
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
# TA / competition tips: 224 often hides axis or map text; 512 aligns ~SmolVLM vision longest_edge.
IMG_SIZE = 224
# Train-only aug (crop / rotation / jitter) — see vqa_dataset.TaScienceVQADataset(image_augment=...)
IMAGE_AUGMENT = False
MAX_NEW_TOKENS_VAL = 1  # SmolVLM/BPE: need several tokens for a clean choice letter; 1 gives nonsense val acc.


def get_hf_token() -> str | None:
    """Hub token only: ``HF_TOKEN`` / ``HUGGING_FACE_HUB_TOKEN`` in the environment."""
    t = (os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or "").strip()
    if t:
        return t
    if IN_COLAB:
        try:
            from google.colab import userdata

            t = str(userdata.get("HF_TOKEN")).strip()
            return t or None
        except Exception:
            return None
    return None


HF_TOKEN = get_hf_token()

from functools import partial

from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader

OUT_DIR = (DATA_ROOT / "outputs").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
VAL_PREDS_CSV = (OUT_DIR / "val_preds_ta.csv").resolve()
LORA_SAVE_DIR = OUT_DIR / "lora_adapter_ta"

from vqa_dataset import load_punnett_allele_csv

PUNNETT_ALLELE_CSV = (DATA_ROOT / "punnett_allele_overrides.csv").resolve()
PUNNETT_ALLELE_OVERRIDES = load_punnett_allele_csv(PUNNETT_ALLELE_CSV)
# Food web: graph + MC lookup neuro-symbolic overrides on val/test (set False to ablate)
FOOD_WEB_SYMBOLIC = True

print(
    "Punnett allele overrides (symbolic hybrid on val/test):",
    len(PUNNETT_ALLELE_OVERRIDES),
    "rows →",
    PUNNETT_ALLELE_CSV,
)
print("Food web symbolic (val/test):", FOOD_WEB_SYMBOLIC)

# Optional: append parsed dominant/recessive line from Hint on Punnett train rows
INCLUDE_PUNNETT_ALLELE_HINT = False

# --- Training / DataLoader (edit here) ---
BATCH_SIZE = 8 if device.type != "mps" else 1
GRAD_ACCUM = 16 if (device.type == "mps" and BATCH_SIZE == 1) else 1
# LoRA/DoRA: 1e-4 can oscillate with strong aug; try 5e-5–1e-4 and watch *epoch-averaged* train loss.
LR = 1e-4
# WEIGHT_DECAY = 0.01
WEIGHT_DECAY = 0
# LoRA cell: CosineAnnealingLR steps once per epoch; floor = LR * LR_MIN_RATIO
# LR_MIN_RATIO = 0.1
LR_MIN_RATIO = 0
EPOCHS = 7
if device.type == "mps":
    _nw = 0
elif IN_COLAB:
    _nw = 4
elif device.type == "cuda":
    _nw = 16
else:
    _nw = 2
NUM_WORKERS = _nw
PERSISTENT_WORKERS = (NUM_WORKERS > 0) and (not IN_COLAB)

VAL_BATCH_SIZE = BATCH_SIZE if device.type == "cuda" else 1

# --- TA prompt A/B: see vqa_dataset.build_prompt_ta ---
TA_INCLUDE_META  = False
TA_ANSWER_SUFFIX = "Answer:"
#TA_INCLUDE_META  = True
#TA_ANSWER_SUFFIX = "The correct answer is:"
# --- Load saved LoRA: set str path or None to use LORA_SAVE_DIR ---
LORA_ADAPTER_FOLDER: str | None = None

# --- Hub model cache: set True to force re-download when loading from Hub ---

HF_FORCE_REDOWNLOAD = False

# --- LoRA fine-tuning: DoRA (PEFT ``LoraConfig(use_dora=True)``) ---
USE_DORA = False  # magnitude + direction; set False for vanilla LoRA
# TA PDF tip #4: attn (q,v) + MLP (gate/up/down) at lower rank; ~<5M trainable with DoRA @ r=8
LORA_R     = 16
LORA_ALPHA = 256
#LORA_TARGET_MODULES = ["gate_proj", "up_proj", "down_proj"]
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
#LORA_TARGET_MODULES = ["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"]


def train_autocast():
    if device.type == "cuda":
        return torch.amp.autocast("cuda", enabled=True)
    if device.type == "mps":
        return torch.amp.autocast("mps", enabled=True, dtype=torch.float16)
    return contextlib.nullcontext()


print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"CSV_DIR      = {CSV_DIR}")
print(f"IMAGE_ROOT   = {IMAGE_ROOT}")
print(f"device = {device}  MODEL_ID = {MODEL_ID}  IMG_SIZE = {IMG_SIZE}  VAL max_new_tokens = {MAX_NEW_TOKENS_VAL}")
print(f"IMAGE_AUGMENT (train) = {IMAGE_AUGMENT}")
print(f"BATCH_SIZE   = {BATCH_SIZE}     GRAD_ACCUM     = {GRAD_ACCUM}  LR = {LR}  EPOCHS = {EPOCHS}")
print(f"WEIGHT_DECAY = {WEIGHT_DECAY}   LR cosine floor ≈ LR * {LR_MIN_RATIO} (per-epoch step in LoRA cell)")
print(f"NUM_WORKERS  = {NUM_WORKERS}     VAL_BATCH_SIZE = {VAL_BATCH_SIZE}  ")
print(f"INCLUDE_META = {TA_INCLUDE_META}  ANSWER_SUFFIX  = {TA_ANSWER_SUFFIX!r}")
print(f"USE_DORA     = {USE_DORA}")
print(f"LoRA r={LORA_R} alpha={LORA_ALPHA} targets={LORA_TARGET_MODULES}")


Mounted at /content/drive


CUDA device: NVIDIA A100-SXM4-80GB  VRAM ~79.3 GiB
Punnett allele overrides (symbolic hybrid on val/test): 187 rows → /content/drive/My Drive/DL/final/data/punnett_allele_overrides.csv
Food web symbolic (val/test): True
PROJECT_ROOT = /content/drive/My Drive/DL/final
CSV_DIR      = /content/drive/My Drive/DL/final/pixels-to-predictions
IMAGE_ROOT   = /content/drive/My Drive/DL/final/pixels-to-predictions/images/images
device = cuda:0  MODEL_ID = HuggingFaceTB/SmolVLM-500M-Instruct  IMG_SIZE = 224  VAL max_new_tokens = 1
IMAGE_AUGMENT (train) = False
BATCH_SIZE   = 8     GRAD_ACCUM     = 1  LR = 0.0001  EPOCHS = 7
WEIGHT_DECAY = 0   LR cosine floor ≈ LR * 0 (per-epoch step in LoRA cell)
NUM_WORKERS  = 4     VAL_BATCH_SIZE = 8  
INCLUDE_META = False  ANSWER_SUFFIX  = 'Answer:'
USE_DORA     = False
LoRA r=16 alpha=256 targets=['q_proj', 'k_proj', 'v_proj', 'o_proj']


In [2]:
import importlib

import vqa_dataset

importlib.reload(vqa_dataset)

from vqa_dataset import (
    TaScienceVQADataset,
    build_prompt_ta,
    extract_ta_choice_letter,
    join_p2p_image_path,
    load_image_rgb_cached,
    parse_choices_cell,
    ta_letter_to_index,
)

train_df = pd.read_csv(CSV_DIR / "train.csv")
val_df = pd.read_csv(CSV_DIR / "val.csv")
test_df = pd.read_csv(CSV_DIR / "test.csv")
for _df in (train_df, val_df, test_df):
    _df["choices"] = _df["choices"].apply(parse_choices_cell)

print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")

Train 3,109 | Val 1,048 | Test 1,008


In [3]:
# Example prompt (with gold letter). Uses ``TA_INCLUDE_META`` / ``TA_ANSWER_SUFFIX`` from the setup cell.
print(
    build_prompt_ta(
        train_df.iloc[8],
        include_answer=True,
        include_meta=TA_INCLUDE_META,
        answer_suffix=TA_ANSWER_SUFFIX,
    )
)

<image>
Context:
Offspring genotypes: homozygous or heterozygous?
How do you determine whether an organism is homozygous or heterozygous for a gene? Look at the alleles in the organism's genotype for that gene.
An organism with two identical alleles for a gene is homozygous for that gene.
If both alleles are dominant, the organism is homozygous dominant for the gene.
If both alleles are recessive, the organism is homozygous recessive for the gene.
An organism with two different alleles for a gene is heterozygous for that gene.
In a Punnett square, each box represents a different outcome, or result. Each of the four outcomes is equally likely to happen. Each box represents one way the parents' alleles can combine to form an offspring's genotype. 
Because there are four boxes in the Punnett square, there are four possible outcomes.
An event is a set of one or more outcomes. The probability of an event is a measure of how likely the event is to happen. This probability is a number between

In [4]:
from vqa_dataset import run_validation_ta_batched


def run_validation_ta(
    model,
    processor,
    val_df: pd.DataFrame,
    *,
    desc: str = "val (TA)",
    save_preds_csv=VAL_PREDS_CSV,
):
    """TA starter metric (full decode + letter parse). Writes ``VAL_PREDS_CSV`` unless ``save_preds_csv`` is None."""
    return run_validation_ta_batched(
        model,
        processor,
        val_df,
        IMAGE_ROOT,
        img_size=IMG_SIZE,
        batch_size=VAL_BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS_VAL,
        num_workers=NUM_WORKERS,
        persistent_workers=PERSISTENT_WORKERS,
        device=device,
        desc=desc,
        include_meta=TA_INCLUDE_META,
        answer_suffix=TA_ANSWER_SUFFIX,
        save_preds_csv=save_preds_csv,
        punnett_allele_overrides=PUNNETT_ALLELE_OVERRIDES,
        food_web_symbolic=FOOD_WEB_SYMBOLIC,
    )


## Load SmolVLM

In [5]:
_hf_kw = {}
if HF_TOKEN:
    _hf_kw["token"] = HF_TOKEN

processor = AutoProcessor.from_pretrained(MODEL_ID, **_hf_kw)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

_dtype = torch.float16 if device.type == "cuda" else torch.float32
_load_kw = dict(dtype=_dtype, low_cpu_mem_usage=True, **_hf_kw)
if device.type == "cuda":
    _load_kw["device_map"] = "auto"
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **_load_kw)
if device.type != "cuda":
    model.to(device)
model.eval()
print("Loaded", MODEL_ID)

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded HuggingFaceTB/SmolVLM-500M-Instruct


## Full validation (starter parity)

Same metric as the TA notebook: **full** `batch_decode` + `extract_ta_choice_letter`.  
**Speed:** validation uses a **batched** `DataLoader` (`VAL_BATCH_SIZE`, default = `BATCH_SIZE` on CUDA, **1** on MPS). The old **1048× separate `generate`** loop was effective batch size **1**, which is much slower on Colab. Override with env **`VAL_BATCH_SIZE=1`** if you need the sequential loop for debugging.

In [6]:
acc, val_preds_all = run_validation_ta(model, processor, val_df, desc="val (all, TA)")
print(f"Val accuracy (all {len(val_preds_all)}): {acc:.6f}")

out_path = DATA_ROOT / "outputs" / "val_preds_ta.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
val_preds_all.to_csv(out_path, index=False)
print("Wrote:", out_path.resolve())

val (all, TA):   0%|          | 0/131 [00:00<?, ?it/s]

Wrote val predictions → /content/drive/My Drive/DL/final/data/outputs/val_preds_ta.csv  rows=1048  acc=0.608779
Val accuracy (all 1048): 0.608779
Wrote: /content/drive/My Drive/DL/final/data/outputs/val_preds_ta.csv


## Optional: `TaScienceVQADataset` smoke test

Use this class with `DataLoader` + `collate_vqa_eval` from `vqa_dataset` if you want batched inference later; for TA parity numbers, keep the **row loop** above.

In [7]:
ds = TaScienceVQADataset(
    val_df.head(4),
    IMAGE_ROOT,
    img_size=IMG_SIZE,
    is_train=False,
    include_meta=TA_INCLUDE_META,
    answer_suffix=TA_ANSWER_SUFFIX,
)
ex = ds[0]
assert "prompt" in ex and "image" in ex
print(ex["id"], ex["prompt"][:200], "...")


val_00671 <image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help th ...


## LoRA fine-tuning (TA prompts)

- **Run** after **Environment** (cell 1) and **Data** (cell 2). The **`run_validation_ta`** cell must have been executed so the function exists (or run cells 1–4 in order).
- **Recommended:** restart the kernel and run **1 → 2 → 4 → this section** only, so the GPU loads a single model path (baseline *or* training).
- Training uses **`TaScienceVQADataset`** + **`collate_vqa_train`**; each epoch ends with **TA val** (same metric as ~0.543 zero-shot).
- Best checkpoint → **`data/outputs/lora_adapter_ta/`**.


In [8]:
# LoRA fine-tuning — TA prompts, TA val each epoch
import importlib

import vqa_dataset as vq

importlib.reload(vq)

from vqa_dataset import (
    TaScienceVQADataset,
    collate_vqa_train,
    set_left_padding_for_decoder_generate,
)

TEXT_ONLY_BASELINE = False

train_ds = TaScienceVQADataset(
    train_df,
    IMAGE_ROOT,
    img_size=IMG_SIZE,
    is_train=True,
    text_only=TEXT_ONLY_BASELINE,
    include_meta=TA_INCLUDE_META,
    answer_suffix=TA_ANSWER_SUFFIX,
    image_augment=IMAGE_AUGMENT,
    include_punnett_allele_hint=INCLUDE_PUNNETT_ALLELE_HINT,
)

_hf_kw_ft: dict = {}
if HF_TOKEN:
    _hf_kw_ft["token"] = HF_TOKEN
if HF_FORCE_REDOWNLOAD:
    _hf_kw_ft["force_download"] = True

processor = AutoProcessor.from_pretrained(MODEL_ID, **_hf_kw_ft)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

if device.type == "cuda" and torch.cuda.is_bf16_supported():
    _model_dtype = torch.bfloat16
elif device.type == "mps":
    _model_dtype = torch.float32
else:
    _model_dtype = torch.float32

_mps_kw: dict = {}
if device.type == "mps":
    _mps_kw["attn_implementation"] = "eager"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=_model_dtype,
    low_cpu_mem_usage=True,
    **_hf_kw_ft,
    **_mps_kw,
).to(device)

set_left_padding_for_decoder_generate(processor)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
    use_dora=USE_DORA,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

_base = model.get_base_model() if hasattr(model, "get_base_model") else model
for cfg in (
    getattr(_base, "config", None),
    getattr(getattr(_base, "config", None), "text_config", None),
    getattr(getattr(_base, "config", None), "vision_config", None),
):
    if cfg is not None and hasattr(cfg, "use_cache"):
        cfg.use_cache = False
# PEFT / frozen backbone: default reentrant checkpoint warns ``requires_grad`` and breaks ``loss.backward``; HF uses ``use_reentrant=False``.\n
if hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        model.gradient_checkpointing_enable()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=PERSISTENT_WORKERS,
    collate_fn=partial(collate_vqa_train, processor),
)


def _to_device_batch(batch, dev: torch.device, train: bool) -> dict:
    out = {}
    non_blocking = dev.type == "cuda"
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(dev, non_blocking=non_blocking)
        else:
            out[k] = v
    if not train:
        out.pop("answers", None)
    return out


optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(EPOCHS, 1), eta_min=LR * LR_MIN_RATIO
)
grad_scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

best_acc = -1.0
best_epoch = 0
model.train()

for epoch in range(EPOCHS):
    pbar = tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)
    step_in_epoch = 0
    loss_sum = 0.0
    loss_n = 0

    for batch in pbar:
        batch = _to_device_batch(batch, device, train=True)
        with train_autocast():
            out = model(
                **{
                    k: v
                    for k, v in batch.items()
                    if k
                    in (
                        "input_ids",
                        "attention_mask",
                        "pixel_values",
                        "pixel_attention_mask",
                        "labels",
                    )
                }
            )
            loss = out.loss / GRAD_ACCUM

        loss_mb = float((loss * GRAD_ACCUM).detach().item())
        loss_sum += loss_mb
        loss_n += 1

        if grad_scaler is not None:
            grad_scaler.scale(loss).backward()
        else:
            loss.backward()
        del out

        step_in_epoch += 1

        if step_in_epoch % GRAD_ACCUM == 0:
            if grad_scaler is not None:
                grad_scaler.step(optimizer)
                grad_scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        pbar.set_postfix(
            loss_last=f"{loss_mb:.3f}",
            loss_avg=f"{(loss_sum / max(loss_n, 1)):.3f}",
        )

    if step_in_epoch % GRAD_ACCUM != 0:
        if grad_scaler is not None:
            grad_scaler.step(optimizer)
            grad_scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    train_loss_avg = loss_sum / max(loss_n, 1)
    acc, _vdf = run_validation_ta(
        model, processor, val_df, desc=f"val ep{epoch + 1}"
    )
    print(
        f"epoch {epoch + 1} TA val acc: {acc:.6f}  |  train CE (epoch mean): {train_loss_avg:.4f}  lr: {scheduler.get_last_lr()[0]:.2e}"
    )
    scheduler.step()

    if acc > best_acc:
        best_acc = acc
        best_epoch = epoch + 1
        LORA_SAVE_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(LORA_SAVE_DIR)
        processor.save_pretrained(LORA_SAVE_DIR)
        print(f"Saved best adapter → {LORA_SAVE_DIR}  acc={best_acc:.6f}")

    model.train()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.empty_cache()

print(f"Best TA val: epoch {best_epoch}  acc={best_acc:.6f}")


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

trainable params: 4,161,536 || all params: 511,643,840 || trainable%: 0.8134


epoch 1/7:   0%|          | 0/389 [00:00<?, ?it/s]

val ep1:   0%|          | 0/131 [00:00<?, ?it/s]

Wrote val predictions → /content/drive/My Drive/DL/final/data/outputs/val_preds_ta.csv  rows=1048  acc=0.777672
epoch 1 TA val acc: 0.777672  |  train CE (epoch mean): 0.5877  lr: 1.00e-04
Saved best adapter → /content/drive/My Drive/DL/final/data/outputs/lora_adapter_ta  acc=0.777672


...

val ep6:   0%|          | 0/131 [00:00<?, ?it/s]

Wrote val predictions → /content/drive/My Drive/DL/final/data/outputs/val_preds_ta.csv  rows=1048  acc=0.903626
epoch 6 TA val acc: 0.903626  |  train CE (epoch mean): 0.0972  lr: 1.88e-05
Saved best adapter → /content/drive/My Drive/DL/final/data/outputs/lora_adapter_ta  acc=0.903626


epoch 7/7:   0%|          | 0/389 [00:00<?, ?it/s]

val ep7:   0%|          | 0/131 [00:00<?, ?it/s]

Wrote val predictions → /content/drive/My Drive/DL/final/data/outputs/val_preds_ta.csv  rows=1048  acc=0.901718
epoch 7 TA val acc: 0.901718  |  train CE (epoch mean): 0.0516  lr: 4.95e-06
Best TA val: epoch 6  acc=0.903626


In [9]:
from peft import PeftModel

from vqa_dataset import set_left_padding_for_decoder_generate

if LORA_ADAPTER_FOLDER:
    LORA_ADAPTER_PATH = Path(LORA_ADAPTER_FOLDER).expanduser().resolve()
else:
    LORA_ADAPTER_PATH = Path(LORA_SAVE_DIR).resolve()

if not LORA_ADAPTER_PATH.is_dir():
    raise FileNotFoundError(
        f"No LoRA folder: {LORA_ADAPTER_PATH}\n"
        "Train first or set LORA_ADAPTER_FOLDER in the setup cell."
    )

_hf_l = {}
if HF_TOKEN:
    _hf_l["token"] = HF_TOKEN

processor = AutoProcessor.from_pretrained(str(LORA_ADAPTER_PATH), **_hf_l)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
set_left_padding_for_decoder_generate(processor)

if device.type == "cuda" and torch.cuda.is_bf16_supported():
    _md_l = torch.bfloat16
elif device.type == "mps":
    _md_l = torch.float32
else:
    _md_l = torch.float32

_mps_kw_l: dict = {}
if device.type == "mps":
    _mps_kw_l["attn_implementation"] = "eager"

base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=_md_l,
    low_cpu_mem_usage=True,
    **_hf_l,
    **_mps_kw_l,
).to(device)
model = PeftModel.from_pretrained(base, str(LORA_ADAPTER_PATH), **_hf_l)
model.eval()
print("Loaded LoRA from", LORA_ADAPTER_PATH)


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Loaded LoRA from /content/drive/My Drive/DL/final/data/outputs/lora_adapter_ta


In [10]:
import importlib

import vqa_dataset

importlib.reload(vqa_dataset)

from vqa_dataset import predict_test_ta_batched

submission_path = DATA_ROOT / "submission.csv"
model.eval()
sub_df = predict_test_ta_batched(
    model,
    processor,
    test_df,
    IMAGE_ROOT,
    img_size=IMG_SIZE,
    batch_size=VAL_BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS_VAL,
    num_workers=NUM_WORKERS,
    persistent_workers=PERSISTENT_WORKERS,
    device=device,
    desc="New_LoRA",
    include_meta=TA_INCLUDE_META,
    answer_suffix=TA_ANSWER_SUFFIX,
    punnett_allele_overrides=PUNNETT_ALLELE_OVERRIDES,
    food_web_symbolic=FOOD_WEB_SYMBOLIC,
)
assert list(sub_df.columns) == ["id", "answer"]
assert sub_df["id"].nunique() == len(sub_df)
submission_path.parent.mkdir(parents=True, exist_ok=True)
sub_df.to_csv(submission_path, index=False)
print("Wrote:", submission_path.resolve(), f"rows={len(sub_df)}")


New_LoRA:   0%|          | 0/126 [00:00<?, ?it/s]

Wrote: /content/drive/My Drive/DL/final/data/submission.csv rows=1008


In [11]:
from google.colab import runtime
runtime.unassign()
